# Task 02: Train and compare models

**Goal:** try three different algorithms on the same problem and compare them honestly.

**Dataset:** Breast Cancer Wisconsin (Diagnostic), bundled with scikit-learn. Binary classification: is a tumour malignant or benign?

**What this notebook does**
1. One train/test split, shared by every model
2. Three different algorithms, plus the trivial baseline from Task 01
3. A comparison table using the **right metric for the problem**, not just accuracy
4. A cross-validation check so I don't over-read a small test set

In [11]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, make_scorer)

RANDOM_STATE = 42

## 1. Load data and split once

In this dataset `target = 0` means **malignant** and `target = 1` means **benign**. I split once, before anything else, and every model below uses these exact same four objects.

In [12]:
df = load_breast_cancer(as_frame=True).frame
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (455, 30) | Test: (114, 30)


## 2. Choosing the metric

Accuracy treats every mistake as equal, but here they are not:

- **Missing a malignant tumour** (false negative) can delay treatment. This is the costly error.
- **Flagging a benign tumour as malignant** (false positive) leads to extra tests. This is a smaller cost.

So the main metric is **recall for the malignant class**: of all truly malignant tumours, how many did the model catch? I also report precision and F1 for the malignant class, plus accuracy for reference.

Because malignant is class `0` here, I pass `pos_label=0` everywhere. Leaving it at the default would silently measure the benign class instead.

## 3. The three models (plus baseline)

I picked three algorithms from different families:

| Model | Idea | Needs scaling? |
|---|---|---|
| Logistic Regression | a linear decision boundary | yes |
| K-Nearest Neighbours (k=5) | vote of the closest training points | yes |
| Random Forest | many decision trees voting together | no |

Models that need scaling get a `StandardScaler` inside a pipeline, so it is fitted on training data only. All use default-style settings with no tuning, so the comparison is about the algorithms and not about who was tuned harder.

In [13]:
models = {
    "Baseline (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

## 4. Train and evaluate on the same test set

For each model I also run 5-fold cross-validation on the **training data only** and record malignant recall. The test set is only 114 rows, so one extra mistake moves recall by a couple of points. The cross-validation number shows how stable each result is.

In [14]:
recall_malignant = make_scorer(recall_score, pos_label=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
matrices = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=recall_malignant)

    matrices[name] = confusion_matrix(y_test, pred)
    rows.append({
        "Model": name,
        "Recall (malignant)": recall_score(y_test, pred, pos_label=0),
        "Precision (malignant)": precision_score(y_test, pred, pos_label=0, zero_division=0),
        "F1 (malignant)": f1_score(y_test, pred, pos_label=0),
        "Accuracy": accuracy_score(y_test, pred),
        "CV recall mean": cv_scores.mean(),
        "CV recall std": cv_scores.std(),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

,Recall (malignant),Precision (malignant),F1 (malignant),Accuracy,CV recall mean,CV recall std
Model,,,,,,
Baseline (most frequent),0.000,0.000,0.000,0.632,0.000,0.000
Logistic Regression,0.976,0.976,0.976,0.982,0.965,0.029
KNN (k=5),0.929,0.951,0.940,0.956,0.929,0.024
Random Forest,0.929,0.951,0.940,0.956,0.953,0.030


The confusion matrices show the actual mistakes (rows = actual, columns = predicted; 0 = malignant, 1 = benign):

In [15]:
for name, cm in matrices.items():
    print(name)
    print(pd.DataFrame(cm, index=["actual malignant", "actual benign"],
                       columns=["pred malignant", "pred benign"]))
    print()

Baseline (most frequent)
                  pred malignant  pred benign
actual malignant               0           42
actual benign                  0           72

Logistic Regression
                  pred malignant  pred benign
actual malignant              41            1
actual benign                  1           71

KNN (k=5)
                  pred malignant  pred benign
actual malignant              39            3
actual benign                  2           70

Random Forest
                  pred malignant  pred benign
actual malignant              39            3
actual benign                  2           70



## 5. Reading the results honestly

- **Every real model beats the baseline by a wide margin.** The baseline gets 63% accuracy but catches **zero** malignant tumours, which is exactly why accuracy alone is a poor guide here.
- **Logistic Regression scores best on the test set**, missing 1 malignant tumour out of 42.
- **KNN and Random Forest tie on the test set**, each missing 3.
- **The gaps are small.** Between Logistic Regression and the other two the difference is 2 missed tumours out of 42. The cross-validation spread (about 0.03) is as large as those gaps, so I would not claim Logistic Regression is truly better. The honest conclusion is that all three work well, and Logistic Regression is a reasonable pick because it is simplest and scored best on both the test set and the cross-validated mean.
- **Cross-validation shows Random Forest is steadier than its test result suggests** (CV recall about 0.95 versus 0.93 on the test set), which is another sign that the test set alone is too small to rank models with confidence.

## Decisions and limitations

- **Metric:** recall on the malignant class, because a missed cancer costs more than a false alarm. Accuracy is shown for reference only.
- **No tuning:** all models use near-default settings so the comparison stays fair.
- **Single split:** the table above uses one split, plus cross-validation on the training set as a stability check. A repeated cross-validation over the whole dataset would be a stronger comparison.
- **Dataset size:** 569 rows is small, so differences of a few percentage points should not be over-interpreted.